# 🌾 Planetary Data Preprocessing & Cleaning Pipeline (Complete)

Ce notebook documente la **Phase de Traitement et de Nettoyage des Données** pour la simulation planétaire. 
Il regroupe les scripts de nettoyage utilisés pour traiter à la fois :
1. **Les données historiques, démographiques et socio-économiques** (FAOSTAT, World Bank, OWID, WHO, ACLED/UCDP).
2. **Les données physiques et géologiques** (volcans, séismes, infrastructures énergétiques, écologie).
3. **Les données d'hydrologie** (lacs, rivières, prélèvements d'eau douce).
4. **Les données raster de climat (WorldClim)** au format GeoTIFF.

Pour chaque dataset, nous explicitons les choix de variables, les justifications scientifiques des filtres et le traitement des valeurs aberrantes (outliers).

## ⚙️ Initialisation du Pipeline

Nous configurons les répertoires et chargeons les dépendances standard. Les données brutes proviennent de `data/raw/` et les données nettoyées seront stockées dans `data/cleaned/`.

In [16]:
import os
import json
import math
import pandas as pd
import numpy as np
from PIL import Image

RAW_DIR = "data/raw"
CLEANED_DIR = "data/cleaned"
os.makedirs(CLEANED_DIR, exist_ok=True)

def load_csv(filepath):
    """Charge un fichier CSV en testant plusieurs encodages courants sous Windows/Linux."""
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(filepath, encoding=enc, low_memory=False, on_bad_lines='skip')
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f"Impossible de lire {filepath}")

def find_col(df, *names):
    # Exact match first
    for name in names:
        if name in df.columns:
            return name
            
    # Case-insensitive match
    for name in names:
        for col in df.columns:
            if name.lower() == col.lower():
                return col
                
    # Normalize unicode/strip accents and non-alphanumeric characters for robust matching
    import unicodedata
    def norm(s):
        s = str(s).lower()
        # strip accents
        s = "".join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
        # strip non-alphanumeric characters (like replacement character \ufffd)
        s = "".join(c for c in s if c.isalnum())
        # common corruptions in these datasets:
        s = s.replace('element', 'lment').replace('annee', 'anne').replace('unite', 'unit')
        return s

    for name in names:
        norm_name = norm(name)
        for col in df.columns:
            if norm_name == norm(col):
                return col

    for name in names:
        norm_name = norm(name)
        for col in df.columns:
            norm_col = norm(col)
            if norm_name in norm_col or norm_col in norm_name:
                # Avoid matching a 'code' column if the search target does not have 'code'
                if 'code' in norm_col and 'code' not in norm_name:
                    continue
                return col
                
    # Fallback to the original matching logic
    for name in names:
        base = name.lower().replace('é', '').replace('è', '').replace('ê', '').replace('à', '')
        for col in df.columns:
            col_clean = col.lower().replace('é', '').replace('è', '').replace('ê', '').replace('à', '')
            if len(base) >= 4 and (base[:4] in col_clean or base[:4] in col.lower()):
                return col
    return None
def clean_common(df, value_col='Valeur'):
    """Nettoie les valeurs manquantes, les doublons et convertit les colonnes clés."""
    df = df.dropna(subset=[value_col])
    df = df.drop_duplicates()
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    df = df.dropna(subset=[value_col])
    
    annee_col = find_col(df, 'Année', 'Annee', 'Year')
    if annee_col:
        df[annee_col] = pd.to_numeric(df[annee_col], errors='coerce')
        df = df.dropna(subset=[annee_col])
        df[annee_col] = df[annee_col].astype(int)
    return df

def save_cleaned(df, name):
    """Sauvegarde le dataset nettoyé dans le dossier cleaned."""
    filepath = os.path.join(CLEANED_DIR, name)
    df.to_csv(filepath, index=False, encoding='utf-8')
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"[Sauvegarde] {name} : {df.shape[0]} lignes, {df.shape[1]} colonnes ({size_mb:.2f} Mo)")

# 🌾 PARTIE 1 : Traitement des données historiques agricoles & climatiques (FAOSTAT)

Cette partie s'attache à harmoniser les échelles et les unités pour que les modèles de Machine Learning puissent apprendre des relations physiques et agronomiques cohérentes sans être biaisés par les différences de superficie entre pays.

### 1️⃣ Bilan Nutritif des Sols & Engrais

*   **Pourquoi ce dataset ?** Permet d'évaluer la quantité de nutriments (Azote N, Phosphore P, Potassium K) entrant et sortant des sols cultivés par pays.
*   **Choix des variables & Unités :** Nous filtrons uniquement sur l'unité **`kg/ha`** (kilogramme par hectare de terre arable). Conserver les volumes globaux en *tonnes* fausserait l'apprentissage, car un grand pays consommerait mécaniquement plus qu'un petit pays sans que cela reflète l'intensité ou la qualité du sol.
*   **Outliers & Nettoyage :** 
    *   Nous trions et séparons en deux fichiers : `bilan_nutritif_sols.csv` (bilan net d'azote/phosphore par hectare) et `fertilizers_nutrient.csv` (intrants azotés chimiques appliqués aux sols).
    *   Les lignes sans valeur de bilan ou sans année valide sont éliminées.

In [17]:
sol_path = os.path.join(RAW_DIR, "pedologie_sols", "sol_nutritif", "Environnement_Bilan_nutritif_des_sols_F_Toutes_les_Données_(Normalisé).csv")

if os.path.exists(sol_path):
    df_sol = load_csv(sol_path)
    
    col_unite = find_col(df_sol, 'Unité', 'Unit')
    col_element = find_col(df_sol, 'Élément', 'Element')
    col_zone = find_col(df_sol, 'Zone', 'Area')
    col_annee = find_col(df_sol, 'Année', 'Year')
    col_valeur = find_col(df_sol, 'Valeur', 'Value')
    col_produit = find_col(df_sol, 'Produit', 'Item')
    
    # 1. Bilan Nutritif Sols (kg/ha)
    df_sols_kgha = df_sol[df_sol[col_unite] == 'kg/ha'].copy()
    cols_keep_sols = {col_zone: 'Pays', col_produit: 'Produit', col_annee: 'Annee', col_valeur: 'Valeur'}
    df_sols_kgha = df_sols_kgha[[c for c in cols_keep_sols.keys()]].copy().rename(columns=cols_keep_sols)
    df_sols_kgha = clean_common(df_sols_kgha)
    save_cleaned(df_sols_kgha, "bilan_nutritif_sols.csv")
    
    # 2. Bilan Nutritif Terres Cultivées (kg/ha + %)
    df_terres = df_sol[df_sol[col_unite].isin(['kg/ha', '%'])].copy()
    cols_keep_terres = {col_zone: 'Pays', col_produit: 'Produit', col_element: 'Element', col_annee: 'Annee', col_unite: 'Unite', col_valeur: 'Valeur'}
    df_terres = df_terres[[c for c in cols_keep_terres.keys()]].copy().rename(columns=cols_keep_terres)
    df_terres = clean_common(df_terres)
    save_cleaned(df_terres, "bilan_nutritif_terres_cultivees.csv")
    
    # 3. Engrais chimiques (intrants d'azote/potasse/phosphate synthétiques en kg/ha)
    df_fert = df_sol[df_sol[col_produit].astype(str).str.contains('Engrais', case=False, na=False)].copy()
    df_fert = df_fert[df_fert[col_unite] == 'kg/ha'].copy()
    cols_keep_fert = {col_zone: 'Pays', col_produit: 'Produit', col_annee: 'Annee', col_valeur: 'Valeur'}
    df_fert = df_fert[[c for c in cols_keep_fert.keys()]].copy().rename(columns=cols_keep_fert)
    df_fert = clean_common(df_fert)
    save_cleaned(df_fert, "fertilizers_nutrient.csv")
else:
    print("Fichier Bilan Nutritif Sols absent, saut de cette étape.")

[Sauvegarde] bilan_nutritif_sols.csv : 96793 lignes, 4 colonnes (4.33 Mo)
[Sauvegarde] bilan_nutritif_terres_cultivees.csv : 96793 lignes, 6 colonnes (11.90 Mo)
[Sauvegarde] fertilizers_nutrient.csv : 10334 lignes, 4 colonnes (0.49 Mo)


### 2️⃣ Variation de Température

*   **Pourquoi ce dataset ?** Il fournit les écarts de température mensuels et annuels observés par rapport à la climatologie de référence.
*   **Choix des variables :** Nous conservons le pays, l'année, le mois (ou indicateur annuel) et la valeur de l'écart en **°C**.
*   **Outliers & Nettoyage :** Suppression des doublons et des relevés sans valeur numérique.

In [18]:
temp_path = os.path.join(RAW_DIR, "atmosphere_climat", "temperature", "Environnement_variation_temperature_F_Toutes_les_Données_(Normalisé).csv")
if os.path.exists(temp_path):
    df_temp = load_csv(temp_path)
    col_zone = find_col(df_temp, 'Zone')
    col_annee = find_col(df_temp, 'Année', 'Annee', 'Year')
    col_valeur = find_col(df_temp, 'Valeur')
    col_element = find_col(df_temp, 'Élément', 'Element')
    col_mois = find_col(df_temp, 'Mois')
    col_code_mois = find_col(df_temp, 'Code Mois')
    
    cols_keep = {col_zone: 'Pays', col_code_mois: 'Code Mois', col_mois: 'Mois', col_element: 'Element', col_annee: 'Annee', col_valeur: 'Valeur'}
    df_temp = df_temp[[c for c in cols_keep.keys()]].copy().rename(columns=cols_keep)
    df_temp = clean_common(df_temp)
    save_cleaned(df_temp, "variation_temperature.csv")
else:
    print("Fichier Variation de Température absent, saut de cette étape.")

[Sauvegarde] variation_temperature.csv : 548204 lignes, 6 colonnes (31.61 Mo)


### 3️⃣ Intrants & Utilisation des Terres

*   **Pourquoi ce dataset ?** Contient les surfaces forestières, prairiales, et irriguées par pays.
*   **Choix des variables & Unités :** Seules les variables exprimées en **`1000 ha`** (surfaces réelles) ou **`%`** (part de territoire) sont conservées.

In [19]:
terres_path = os.path.join(RAW_DIR, "socio_economie_demographie", "faostat", "terres_utilisation", "Intrants_TerresUtilisation_F_Toutes_les_Données_(Normalisé).csv")
if os.path.exists(terres_path):
    df_terres_util = load_csv(terres_path)
    col_unite = find_col(df_terres_util, 'Unité', 'Unite')
    col_element = find_col(df_terres_util, 'Élément', 'Element')
    col_zone = find_col(df_terres_util, 'Zone')
    col_annee = find_col(df_terres_util, 'Année', 'Annee', 'Year')
    col_valeur = find_col(df_terres_util, 'Valeur')
    col_produit = find_col(df_terres_util, 'Produit')
    
    df_terres_util = df_terres_util[df_terres_util[col_unite].isin(['1000 ha', '%'])].copy()
    cols_keep = {col_zone: 'Pays', col_produit: 'Produit', col_element: 'Element', col_annee: 'Annee', col_unite: 'Unite', col_valeur: 'Valeur'}
    df_terres_util = df_terres_util[[c for c in cols_keep.keys()]].copy().rename(columns=cols_keep)
    df_terres_util = clean_common(df_terres_util)
    save_cleaned(df_terres_util, "intrants_utilisation_terres.csv")
else:
    print("Fichier Utilisation des Terres absent, saut de cette étape.")

[Sauvegarde] intrants_utilisation_terres.csv : 374594 lignes, 6 colonnes (26.20 Mo)


### 4️⃣ Production Cultures et Animaux

*   **Pourquoi ce dataset ?** Contient les rendements agricoles et les cheptels d'élevage mondiaux.
*   **Choix des variables & Unités :** 
    *   *Cultures* : rendement en **`kg/ha`** (mesure d'efficacité biologique), production en **`tonnes`**, superficie en **`ha`**.
    *   *Animaux* : effectifs en têtes de bétail ou production de viande en tonnes.

In [20]:
prod_path = os.path.join(RAW_DIR, "socio_economie_demographie", "faostat", "production_agricole", "Production_Cultures_ProduitsAnimaux_F_Toutes_les_Données_(Normalisé).csv")
if os.path.exists(prod_path):
    df_prod = load_csv(prod_path)
    col_unite = find_col(df_prod, 'Unité', 'Unite')
    col_element = find_col(df_prod, 'Élément', 'Element')
    col_zone = find_col(df_prod, 'Zone')
    col_annee = find_col(df_prod, 'Année', 'Annee', 'Year')
    col_valeur = find_col(df_prod, 'Valeur')
    col_produit = find_col(df_prod, 'Produit')
    
    mask_cultures = (
        df_prod[col_element].isin(['Production', 'Rendement', 'Superficie récoltée']) &
        df_prod[col_unite].isin(['tonnes', 'kg/ha', 'ha'])
    )
    
    cols_keep = {col_zone: 'Pays', col_produit: 'Produit', col_element: 'Element', col_annee: 'Annee', col_unite: 'Unite', col_valeur: 'Valeur'}
    
    df_cultures = df_prod[mask_cultures].copy()
    df_cultures = df_cultures[[c for c in cols_keep.keys()]].copy().rename(columns=cols_keep)
    df_cultures = clean_common(df_cultures)
    save_cleaned(df_cultures, "production_cultures.csv")
    
    df_animaux = df_prod[~mask_cultures].copy()
    df_animaux = df_animaux[[c for c in cols_keep.keys()]].copy().rename(columns=cols_keep)
    df_animaux = clean_common(df_animaux)
    save_cleaned(df_animaux, "production_animaux.csv")
else:
    print("Fichier Production Agricole absent, saut de cette étape.")

[Sauvegarde] production_cultures.csv : 3251455 lignes, 6 colonnes (223.47 Mo)
[Sauvegarde] production_animaux.csv : 863300 lignes, 6 colonnes (71.61 Mo)


### 5️⃣ Pesticides

*   **Pourquoi ce dataset ?** Quantité de produits phytosanitaires utilisés dans l'agriculture par pays.
*   **Choix des variables :** Nous sélectionnons uniquement les intensités d'utilisation (utilisation par surface cultivée) pour obtenir un indicateur de pression environnementale comparable (exprimé en **`kg/ha`**).

In [21]:
pest_path = os.path.join(RAW_DIR, "socio_economie_demographie", "faostat", "pesticides", "Environnement_Pesticides_F_Toutes_les_Données_(Normalisé).csv")
if os.path.exists(pest_path):
    df_pest = load_csv(pest_path)
    col_zone = find_col(df_pest, 'Zone')
    col_annee = find_col(df_pest, 'Année', 'Year')
    col_valeur = find_col(df_pest, 'Valeur')
    col_produit = find_col(df_pest, 'Produit')
    col_element = find_col(df_pest, 'Élément', 'Element')
    
    df_pest = df_pest[df_pest[col_element].astype(str).str.contains('surface|capita|habit', case=False, na=False)].copy()
    cols_keep = {col_zone: 'Pays', col_produit: 'Produit', col_annee: 'Annee', col_valeur: 'Valeur'}
    df_pest = df_pest[[c for c in cols_keep.keys()]].copy().rename(columns=cols_keep)
    df_pest = clean_common(df_pest)
    save_cleaned(df_pest, "pesticides.csv")
else:
    print("Fichier Pesticides absent, saut de cette étape.")

[Sauvegarde] pesticides.csv : 5214 lignes, 4 colonnes (0.21 Mo)


# 🌋 PARTIE 2 : Traitement des données physiques, géologiques et écologiques

Cette partie traite les datasets localisés par coordonnées GPS (Latitude/Longitude). Le but est de nettoyer les anomalies de saisie, supprimer les relevés n'ayant pas de coordonnées valides, et éliminer les aberrations physiques (altitudes ou profondeurs géologiques impossibles).

### 1️⃣ Volcans Actifs (`volcanoes.csv`)

*   **Pourquoi ce dataset ?** Permet de modéliser le risque volcanique et les flux géothermiques.
*   **Choix des variables :** `volcano_name`, `latitude`, `longitude`, `elevation`, `primary_volcano_type` et la population à proximité.
*   **Outliers & Nettoyage :** 
    *   Nous supprimons les lignes n'ayant pas de coordonnées GPS.
    *   **Filtrage des altitudes aberrantes :** Nous imposons $-5000\text{ m} \le \text{elevation} \le 7000\text{ m}$. Les valeurs hors de cette plage sont des erreurs de saisie car aucun volcan terrestre n'excède 7000 m ou n'est plus profond que 5000 m sous l'eau.

In [22]:
volc_path = os.path.join(RAW_DIR, "geologie", "volcanoes.csv")
if os.path.exists(volc_path):
    df_v = load_csv(volc_path)
    df_v = df_v.dropna(subset=['latitude', 'longitude'])
    
    cols_to_keep = ['volcano_number', 'volcano_name', 'primary_volcano_type', 
                    'last_eruption_year', 'latitude', 'longitude', 'elevation', 'population_within_100_km']
    df_v = df_v[[c for c in cols_to_keep if c in df_v.columns]].copy()
    
    df_v['elevation'] = pd.to_numeric(df_v['elevation'], errors='coerce')
    df_v = df_v[(df_v['elevation'] >= -5000) & (df_v['elevation'] <= 7000)]
    
    df_v['last_eruption_year'] = pd.to_numeric(df_v['last_eruption_year'], errors='coerce')
    save_cleaned(df_v, "volcanoes_cleaned.csv")
else:
    print("Fichier volcanoes.csv absent, saut de cette étape.")

[Sauvegarde] volcanoes_cleaned.csv : 958 lignes, 8 colonnes (0.06 Mo)


### 2️⃣ Séismes Majeurs (`earthquakes_usgs.csv`)

*   **Pourquoi ce dataset ?** Modélisation de l'aléa sismique mondial et des limites de plaques tectoniques.
*   **Choix des variables :** Temps, latitude, longitude, magnitude (`mag`), profondeur de l'hypocentre (`depth`), et localisation.
*   **Outliers & Nettoyage :** 
    *   **Filtrage magnitude :** Strictement supérieure à $0$ et inférieure ou égale à $10$ (la limite physique de l'échelle de Richter/Magnitude de Moment).
    *   **Filtrage profondeur hypocentre :** Strictement positive et inférieure à $800\text{ km}$ (la zone de transition du manteau supérieur limite la sismicité à max 700-800 km).

In [23]:
eq_path = os.path.join(RAW_DIR, "geologie", "earthquakes_usgs.csv")
if os.path.exists(eq_path):
    df_eq = load_csv(eq_path)
    df_eq = df_eq.dropna(subset=['latitude', 'longitude', 'mag'])
    
    cols_to_keep = ['time', 'latitude', 'longitude', 'depth', 'mag', 'place', 'type']
    df_eq = df_eq[[c for c in cols_to_keep if c in df_eq.columns]].copy()
    
    df_eq['mag'] = pd.to_numeric(df_eq['mag'], errors='coerce')
    df_eq['depth'] = pd.to_numeric(df_eq['depth'], errors='coerce')
    df_eq = df_eq[(df_eq['mag'] > 0) & (df_eq['mag'] <= 10)]
    df_eq = df_eq[(df_eq['depth'] >= 0) & (df_eq['depth'] <= 800)]
    
    save_cleaned(df_eq, "earthquakes_cleaned.csv")
else:
    print("Fichier earthquakes_usgs.csv absent, saut de cette étape.")

[Sauvegarde] earthquakes_cleaned.csv : 19878 lignes, 7 colonnes (1.81 Mo)


### 3️⃣ Centrales Électriques Globales (`global_power_plants.csv`)

*   **Pourquoi ce dataset ?** Évaluation de la répartition des ressources énergétiques et de la capacité de production anthropique.
*   **Choix des variables :** Pays, nom, capacité (MW), type de combustible, coordonnées et année de mise en service.
*   **Outliers & Nettoyage :** 
    *   **Filtrage capacité :** Strictement positive (une centrale doit pouvoir produire de l'énergie).
    *   **Filtrage année de mise en service :** Comprise strictement entre $1800$ (début de l'ère industrielle) et $2026$.

In [24]:
pp_path = os.path.join(RAW_DIR, "geologie", "global_power_plants.csv")
if os.path.exists(pp_path):
    df_pp = load_csv(pp_path)
    df_pp = df_pp.dropna(subset=['latitude', 'longitude', 'capacity_mw', 'primary_fuel'])
    
    cols_to_keep = ['country_long', 'name', 'capacity_mw', 'latitude', 'longitude', 'primary_fuel', 'commissioning_year']
    df_pp = df_pp[[c for c in cols_to_keep if c in df_pp.columns]].copy()
    
    df_pp['capacity_mw'] = pd.to_numeric(df_pp['capacity_mw'], errors='coerce')
    df_pp = df_pp[df_pp['capacity_mw'] > 0]
    
    df_pp['commissioning_year'] = pd.to_numeric(df_pp['commissioning_year'], errors='coerce')
    df_pp.loc[(df_pp['commissioning_year'] < 1800) | (df_pp['commissioning_year'] > 2026), 'commissioning_year'] = np.nan
    
    save_cleaned(df_pp, "global_power_plants_cleaned.csv")
else:
    print("Fichier global_power_plants.csv absent, saut de cette étape.")

[Sauvegarde] global_power_plants_cleaned.csv : 34936 lignes, 7 colonnes (2.12 Mo)


### 4️⃣ Densité de Bois Écologique (`wood_density.csv`)

*   **Pourquoi ce dataset ?** Utilisé pour calculer la biomasse forestière globale et le stockage du carbone (Modèles de Miami et Hatton).
*   **Choix des variables :** Espèce, coordonnées, forme de croissance, et densité de bois.
*   **Outliers & Nettoyage :** 
    *   Le fichier est délimité par des points-virgules (`;`) et codé en `latin-1`. Les séparateurs décimaux de type virgule `,` sont remplacés par des points `.` pour le typage numérique.
    *   **Filtrage densité de bois :** Comprise entre $0.1\text{ g/cm}^3$ (bois extrêmement spongieux) et $1.5\text{ g/cm}^3$ (limite biologique supérieure).

In [25]:
wd_path = os.path.join(RAW_DIR, "ecologie_biomasse", "wood_density.csv")
if os.path.exists(wd_path):
    df_wd = pd.read_csv(wd_path, sep=';', encoding='latin-1')
    
    for col in ['Latitude', 'Longitude', 'Wood Density']:
        if col in df_wd.columns:
            df_wd[col] = df_wd[col].astype(str).str.replace(',', '.')
            df_wd[col] = pd.to_numeric(df_wd[col], errors='coerce')
            
    df_wd = df_wd.dropna(subset=['Latitude', 'Longitude', 'Wood Density'])
    
    cols_to_keep = ['Species', 'Latitude', 'Longitude', 'Plant Growth Form', 'Wood Density']
    df_wd = df_wd[[c for c in cols_to_keep if c in df_wd.columns]].copy()
    
    df_wd = df_wd[(df_wd['Wood Density'] >= 0.1) & (df_wd['Wood Density'] <= 1.5)]
    
    save_cleaned(df_wd, "wood_density_cleaned.csv")
else:
    print("Fichier wood_density.csv absent, saut de cette étape.")

[Sauvegarde] wood_density_cleaned.csv : 1159 lignes, 5 colonnes (0.06 Mo)


### 5️⃣ Nouveaux Datasets Géologiques (Minerais, Charbon, Fer, Pétrole & Gaz)

*   **Pourquoi ces datasets ?** Fournit la répartition géographique des gisements miniers et énergétiques mondiaux, indispensables pour simuler les chaînes industrielles de la civilisation.
*   **Nettoyage appliqué :**
    *   *MRDS (USGS)* : Chargement, exclusion des sites sans coordonnées GPS valides, renommage des colonnes.
    *   *Charbon (Global Coal Mine Tracker)* : Chargement de l'onglet `Non-closed mines`, extraction des coordonnées, capacités, et filtrage.
    *   *Fer (Global Iron Ore Mines Tracker)* : Extraction et parsing des coordonnées textuelles `Latitude, Longitude` à partir de la colonne `Coordinates`.
    *   *Pétrole et Gaz (Global Oil & Gas Extraction Tracker)* : Chargement de l'onglet `Field-level main data`, filtrage.


In [26]:
# 1. USGS MRDS Minerals
print("Nettoyage mrds.csv...")
mrds_path = os.path.join(RAW_DIR, "mrds.csv")
if os.path.exists(mrds_path):
    df_mrds = pd.read_csv(mrds_path, low_memory=False)
    cols = ['site_name', 'latitude', 'longitude', 'commod1', 'prod_size', 'dev_stat']
    df_mrds = df_mrds[[c for c in cols if c in df_mrds.columns]].copy()
    df_mrds = df_mrds.dropna(subset=['latitude', 'longitude'])
    df_mrds = df_mrds.drop_duplicates()
    df_mrds = df_mrds.rename(columns={
        'site_name': 'Nom',
        'latitude': 'Latitude',
        'longitude': 'Longitude',
        'commod1': 'Commodite',
        'prod_size': 'Taille_Production',
        'dev_stat': 'Statut_Developpement'
    })
    save_cleaned(df_mrds, "mrds_cleaned.csv")
else:
    print("Fichier mrds.csv absent.")

# 2. Mines de charbon
print("Nettoyage mines de charbon...")
coal_path = os.path.join(RAW_DIR, "geologie", "Global Coal Mine Tracker, May 2026__.xlsx")
if os.path.exists(coal_path):
    df_coal = pd.read_excel(coal_path, sheet_name="Non-closed mines")
    df_coal = df_coal.dropna(subset=['Latitude', 'Longitude'])
    cols_keep = {
        'GEM Mine ID': 'Mine_ID',
        'Mine Name': 'Nom',
        'Country / Area': 'Pays',
        'Status': 'Statut',
        'Capacity (Mtpa)': 'Capacite_Mtpa',
        'Production (Mtpa)': 'Production_Mtpa',
        'Coal Type': 'Type_Charbon',
        'Latitude': 'Latitude',
        'Longitude': 'Longitude'
    }
    df_coal = df_coal[[c for c in cols_keep.keys() if c in df_coal.columns]].rename(columns=cols_keep)
    df_coal = df_coal.drop_duplicates()
    save_cleaned(df_coal, "coal_mines_cleaned.csv")
else:
    print("Tracker mines de charbon absent.")

# 3. Mines de fer
print("Nettoyage mines de fer...")
iron_path = os.path.join(RAW_DIR, "geologie", "Global-Iron-Ore-Mines-Tracker-August-2025-V1.xlsx")
if os.path.exists(iron_path):
    df_iron = pd.read_excel(iron_path, sheet_name="Main Data")
    df_iron = df_iron.dropna(subset=['Coordinates'])
    
    def parse_coords(coord_str):
        try:
            parts = str(coord_str).split(",")
            if len(parts) == 2:
                return float(parts[0].strip()), float(parts[1].strip())
        except Exception:
            pass
        return np.nan, np.nan
        
    coords = df_iron['Coordinates'].apply(parse_coords)
    df_iron['Latitude'] = [c[0] for c in coords]
    df_iron['Longitude'] = [c[1] for c in coords]
    df_iron = df_iron.dropna(subset=['Latitude', 'Longitude'])
    
    cols_keep = {
        'GEM Asset ID': 'Asset_ID',
        'Asset name (English)': 'Nom',
        'Country/Area': 'Pays',
        'Operating status': 'Statut',
        'Production 2024 (ttpa)': 'Production_2024_ttpa',
        'Design capacity (ttpa)': 'Capacite_ttpa',
        'Latitude': 'Latitude',
        'Longitude': 'Longitude'
    }
    df_iron = df_iron[[c for c in cols_keep.keys() if c in df_iron.columns]].rename(columns=cols_keep)
    df_iron = df_iron.drop_duplicates()
    save_cleaned(df_iron, "iron_mines_cleaned.csv")
else:
    print("Tracker mines de fer absent.")

# 4. Pétrole et Gaz
print("Nettoyage pétrole et gaz...")
oil_path = os.path.join(RAW_DIR, "geologie", "Global-Oil-and-Gas-Extraction-Tracker-March-2026.xlsx")
if os.path.exists(oil_path):
    df_oil = pd.read_excel(oil_path, sheet_name="Field-level main data")
    df_oil = df_oil.dropna(subset=['Latitude', 'Longitude'])
    cols_keep = {
        'Unit ID': 'Unit_ID',
        'Unit Name': 'Nom',
        'Fuel type': 'Type_Combustible',
        'Country/Area': 'Pays',
        'Production Type': 'Type_Production',
        'Status': 'Statut',
        'Latitude': 'Latitude',
        'Longitude': 'Longitude'
    }
    df_oil = df_oil[[c for c in cols_keep.keys() if c in df_oil.columns]].rename(columns=cols_keep)
    df_oil = df_oil.drop_duplicates()
    save_cleaned(df_oil, "oil_gas_cleaned.csv")
else:
    print("Tracker pétrole et gaz absent.")


Nettoyage mrds.csv...
[Sauvegarde] mrds_cleaned.csv : 303911 lignes, 6 colonnes (18.29 Mo)
Nettoyage mines de charbon...
[Sauvegarde] coal_mines_cleaned.csv : 5378 lignes, 9 colonnes (0.45 Mo)
Nettoyage mines de fer...
[Sauvegarde] iron_mines_cleaned.csv : 949 lignes, 8 colonnes (0.08 Mo)
Nettoyage pétrole et gaz...
[Sauvegarde] oil_gas_cleaned.csv : 7055 lignes, 8 colonnes (0.84 Mo)


# 💧 PARTIE 3 : Hydrologie & Datasets Socio-Économiques

Cette partie documente le traitement de tous les fichiers d'hydrologie ainsi que des nombreux indicateurs socio-démographiques complémentaires (Banque Mondiale, Our World in Data, UCDP/ACLED et Organisation Mondiale de la Santé).

### 1️⃣ Données d'Hydrologie (`wb_freshwater_withdrawal_pct.csv` et geojson)

*   **Pourquoi ces datasets ?** Permettent d'estimer les prélèvements annuels d'eau douce par pays et de calculer la distance géodésique aux cours d'eau/lacs.
*   **Nettoyage appliqué :** 
    *   *CSV prélèvements* : Standardisation de `country_name` -> `Pays`, `year` -> `Annee` et `value` -> `Valeur` (exprimé en %). Suppression des NaN et doublons.
    *   *GeoJSON lacs/rivières* : Lecture et vérification de la validité des coordonnées de géométrie.

In [27]:
# 1. Prélèvements d'eau douce (Banque Mondiale)
hydro_path = os.path.join(RAW_DIR, "hydrologie_eau", "wb_freshwater_withdrawal_pct.csv")
if os.path.exists(hydro_path):
    df_h = load_csv(hydro_path)
    col_zone = find_col(df_h, 'country_name', 'country_long', 'Zone', 'Area')
    col_annee = find_col(df_h, 'year', 'Year', 'Année')
    col_valeur = find_col(df_h, 'value', 'Value', 'Valeur')
    
    cols_keep = {col_zone: 'Pays', col_annee: 'Annee', col_valeur: 'Valeur'}
    df_h = df_h[[c for c in cols_keep.keys()]].copy().rename(columns=cols_keep)
    df_h = clean_common(df_h)
    save_cleaned(df_h, "wb_freshwater_withdrawal_pct.csv")
else:
    print("wb_freshwater_withdrawal_pct.csv absent.")

# 2. Chargement test des GeoJSON d'hydrologie
for geo_name in ['rivers.geojson', 'lakes.geojson']:
    geo_path = os.path.join(RAW_DIR, "hydrologie_eau", geo_name)
    if os.path.exists(geo_path):
        with open(geo_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"[Vérification GeoJSON] {geo_name} chargé avec succès ({len(data.get('features', []))} entités)")

[Sauvegarde] wb_freshwater_withdrawal_pct.csv : 6592 lignes, 3 colonnes (0.21 Mo)
[Vérification GeoJSON] rivers.geojson chargé avec succès (13 entités)
[Vérification GeoJSON] lakes.geojson chargé avec succès (24 entités)


### 2️⃣ Indicateurs Socio-Économiques Mondiaux

Pour assurer la cohérence et l'homogénéité de nos données, nous nettoyons récursivement tous les sous-dossiers socio-économiques.

*   **World Bank (28 indicateurs)** : GDP, électricité, population, espérance de vie, chômage, éducation...
    *   *Nettoyage* : Les fichiers ont le schéma standard `country_name`, `year`, `value`. Nous les mappons systématiquement vers `Pays`, `Annee`, `Valeur`.
*   **Our World in Data (OWID)** : HDI (Human Development Index), émissions de CO2, pauvreté...
    *   *Nettoyage* : Les fichiers OWID contiennent typiquement `Entity` (colonne 0), `Year` (colonne 2) et l'indicateur numérique (colonne 3). Nous standardisons ces trois colonnes.
*   **Conflits Armés (ACLED/UCDP)** : Intensité et type de conflits historiques par pays.
    *   *Nettoyage* : Nous mappons `location` -> `Pays`, `year` -> `Annee` et `intensity_level` -> `Valeur`.
*   **Organisation Mondiale de la Santé (WHO)** : Mortalité infantile, famine et espérance de vie.

In [28]:
# 1. Nettoyage des 28 fichiers de la Banque Mondiale
wb_dir = os.path.join(RAW_DIR, "socio_economie_demographie", "worldbank")
if os.path.exists(wb_dir):
    for filename in os.listdir(wb_dir):
        if filename.endswith(".csv"):
            df_wb = load_csv(os.path.join(wb_dir, filename))
            col_zone = find_col(df_wb, 'country_name', 'Zone', 'Area')
            col_annee = find_col(df_wb, 'year', 'Year', 'Année')
            col_valeur = find_col(df_wb, 'value', 'Value', 'Valeur')
            if col_zone and col_annee and col_valeur:
                df_wb = df_wb[[col_zone, col_annee, col_valeur]].copy().rename(columns={col_zone:'Pays', col_annee:'Annee', col_valeur:'Valeur'})
                df_wb = clean_common(df_wb)
                save_cleaned(df_wb, filename)

# 2. Nettoyage des fichiers OWID
owid_dir = os.path.join(RAW_DIR, "socio_economie_demographie", "owid")
if os.path.exists(owid_dir):
    for filename in os.listdir(owid_dir):
        if filename.endswith(".csv"):
            df_ow = load_csv(os.path.join(owid_dir, filename))
            col_zone = find_col(df_ow, 'Entity', 'country', 'Zone', 'Area')
            col_annee = find_col(df_ow, 'Year', 'year', 'Année', 'Annee')
            col_valeur = None
            for c in df_ow.columns:
                if c not in [col_zone, col_annee, 'Code', 'code', 'iso_code', 'reporting_level', 'welfare_type', 'ppp_version', 'survey_year', 'survey_comparability']:
                    col_valeur = c
                    break
            if col_zone and col_annee and col_valeur:
                df_ow = df_ow[[col_zone, col_annee, col_valeur]].copy().rename(columns={col_zone:'Pays', col_annee:'Annee', col_valeur:'Valeur'})
                df_ow = clean_common(df_ow)
                save_cleaned(df_ow, filename)
            else:
                print(f"  Skipping {filename} - columns not found")

# 3. Nettoyage d'ACLED / UCDP
acled_path = os.path.join(RAW_DIR, "socio_economie_demographie", "acled", "UcdpPrioConflict_v24_1.csv")
if os.path.exists(acled_path):
    df_ac = load_csv(acled_path)
    col_zone = find_col(df_ac, 'location', 'Zone')
    col_annee = find_col(df_ac, 'year', 'Year')
    col_valeur = find_col(df_ac, 'intensity_level', 'Value')
    if col_zone and col_annee and col_valeur:
        df_ac = df_ac[[col_zone, col_annee, col_valeur]].copy().rename(columns={col_zone:'Pays', col_annee:'Annee', col_valeur:'Valeur'})
        df_ac = clean_common(df_ac)
        save_cleaned(df_ac, "UcdpPrioConflict_v24_1.csv")

# 4. Nettoyage des fichiers de la WHO
who_dir = os.path.join(RAW_DIR, "socio_economie_demographie", "who")
if os.path.exists(who_dir):
    for filename in os.listdir(who_dir):
        if filename.endswith(".csv"):
            df_who = load_csv(os.path.join(who_dir, filename))
            col_zone = find_col(df_who, 'Entity', 'country_name', 'Zone')
            col_annee = find_col(df_who, 'Year', 'year')
            col_valeur = find_col(df_who, 'Value', 'value', 'Valeur', df_who.columns[3] if len(df_who.columns) >= 4 else None)
            if col_zone and col_annee and col_valeur:
                df_who = df_who[[col_zone, col_annee, col_valeur]].copy().rename(columns={col_zone:'Pays', col_annee:'Annee', col_valeur:'Valeur'})
                df_who = clean_common(df_who)
                save_cleaned(df_who, filename)

[Sauvegarde] wb_agricultural_land_pct.csv : 15097 lignes, 3 colonnes (0.52 Mo)
[Sauvegarde] wb_birth_rate.csv : 17195 lignes, 3 colonnes (0.46 Mo)
[Sauvegarde] wb_child_mortality.csv : 13541 lignes, 3 colonnes (0.32 Mo)
[Sauvegarde] wb_deaths_communicable_disease.csv : 1398 lignes, 3 colonnes (0.05 Mo)
[Sauvegarde] wb_death_rate.csv : 17195 lignes, 3 colonnes (0.45 Mo)
[Sauvegarde] wb_electricity_access_pct.csv : 7859 lignes, 3 colonnes (0.21 Mo)
[Sauvegarde] wb_energy_use_per_capita.csv : 6722 lignes, 3 colonnes (0.24 Mo)
[Sauvegarde] wb_gdp_current_usd.csv : 14561 lignes, 3 colonnes (0.51 Mo)
[Sauvegarde] wb_gdp_per_capita.csv : 14561 lignes, 3 colonnes (0.51 Mo)
[Sauvegarde] wb_gini_index.csv : 2430 lignes, 3 colonnes (0.05 Mo)
[Sauvegarde] wb_health_expenditure_gdp.csv : 5716 lignes, 3 colonnes (0.17 Mo)
[Sauvegarde] wb_hiv_prevalence.csv : 5989 lignes, 3 colonnes (0.14 Mo)
[Sauvegarde] wb_hospital_beds_per_1000.csv : 5865 lignes, 3 colonnes (0.16 Mo)
[Sauvegarde] wb_inflation_cpi.

# 🌍 PARTIE 4 : Vérification et lecture des datasets Raster (.tif) de WorldClim

Les données WorldClim ne subissent pas de traitement tabulaire (ce sont des rasters binaires géoréférencés). Cependant, pour s'assurer que les fichiers ne sont pas corrompus et que le moteur spatial `Layer1Engine` les lira correctement, nous exécutons une routine d'inspection sur les en-têtes et métadonnées de chaque catégorie de GeoTIFF.

In [29]:
wc_dir = os.path.join(RAW_DIR, "WorldClim")
if os.path.exists(wc_dir):
    print("Inspection des répertoires WorldClim (.tif) :")
    for folder in sorted(os.listdir(wc_dir)):
        folder_path = os.path.join(wc_dir, folder)
        if os.path.isdir(folder_path):
            tifs = [f for f in os.listdir(folder_path) if f.endswith('.tif')]
            if tifs:
                sample_path = os.path.join(folder_path, tifs[0])
                try:
                    with Image.open(sample_path) as img:
                        width, height = img.size
                        mode = img.mode
                        print(f"  [OK] {folder} (Exemple: {tifs[0]}) : Résolution = {width}x{height}, Mode de pixels = {mode}")
                except Exception as e:
                    print(f"  [ERREUR] Impossible de charger {tifs[0]} dans {folder} : {e}")
            else:
                print(f"  [Vide] Aucun fichier .tif trouvé dans le répertoire {folder}")

Inspection des répertoires WorldClim (.tif) :
  [OK] biovar (Exemple: wc2.1_10m_bio_1.tif) : Résolution = 2160x1080, Mode de pixels = F
  [OK] elev (Exemple: wc2.1_10m_elev.tif) : Résolution = 2160x1080, Mode de pixels = I
  [OK] prec (Exemple: wc2.1_5m_prec_01.tif) : Résolution = 4320x2160, Mode de pixels = I
  [OK] solrad (Exemple: wc2.1_5m_srad_01.tif) : Résolution = 4320x2160, Mode de pixels = I;16
  [OK] tmax (Exemple: wc2.1_5m_tmax_01.tif) : Résolution = 4320x2160, Mode de pixels = F
  [OK] tmin (Exemple: wc2.1_5m_tmin_01.tif) : Résolution = 4320x2160, Mode de pixels = F
  [OK] tvag (Exemple: wc2.1_5m_tavg_01.tif) : Résolution = 4320x2160, Mode de pixels = F
  [OK] vapr (Exemple: wc2.1_5m_vapr_01.tif) : Résolution = 4320x2160, Mode de pixels = F
  [OK] wind (Exemple: wc2.1_5m_wind_01.tif) : Résolution = 4320x2160, Mode de pixels = F


# 🌊 Étape 8 & 9 : Marées & Astronomie (Vérification Algorithmique)

Les deux dernières étapes ne reposent pas sur le nettoyage de fichiers de données brutes tabulaires statiques :
1. **Marées et Dynamique Océanique** : Intégrées à l'aide de la bibliothèque open-source `pyTMD` (Tidal Model Data) pour modéliser l'amplitude de la marée d'équilibre.
2. **Astronomie, Photopériode et Saisons** : Calculées dynamiquement par les équations physiques de déclinaison solaire et d'angle horaire pour obtenir la photopériode (durée du jour) en fonction de la latitude.

Nous validons ci-dessous le bon fonctionnement des paquets et des modèles mathématiques associés.


In [ ]:
import numpy as np
import xarray as xr
import pyTMD.predict
import math

# 1. Validation de pyTMD (Marées)
print("Validation de pyTMD (Marées)...")
t = np.linspace(0.0, 1.0, 24)
ds = xr.Dataset(coords={'y': [48.8566], 'x': [2.3522]})
tide_series = pyTMD.predict.equilibrium_tide(t, ds)
eq_range = float(tide_series.values.max() - tide_series.values.min())
print(f"  [OK] Amplitude de marée d'équilibre calculée pour Paris : {eq_range*1000:.3f} mm")

# 2. Validation des formules de photopériode (Astronomie)
print("Validation de la photopériode...")
def get_photoperiod(lat, doy):
    phi = math.radians(lat)
    delta = 0.409 * math.sin(2.0 * math.pi * (doy - 80.0) / 365.0)
    val = -math.tan(phi) * math.tan(delta)
    if val >= 1.0:
        return 0.0
    elif val <= -1.0:
        return 24.0
    else:
        h_s = math.acos(val)
        return (24.0 / math.pi) * h_s

paris_summer = get_photoperiod(48.8566, 172)
paris_winter = get_photoperiod(48.8566, 355)
print(f"  [OK] Photopériode de Paris - Solstice d'été : {paris_summer:.2f} heures | Solstice d'hiver : {paris_winter:.2f} heures")


Validation de pyTMD (Marées)...
  [OK] Amplitude de marée d'équilibre calculée pour Paris : 2.449 mm
Validation de la photopériode...
  [OK] Photopériode de Paris - Solstice d'été : 15.97 heures | Solstice d'hiver : 8.04 heures


: 

## 🎯 Conclusion Générale du Nettoyage

Tous les fichiers tabulaires de toutes les catégories physiques et socio-économiques (y compris l'hydrologie et World Bank/OWID) sont désormais nettoyés et stockés dans `data/cleaned/`.
Les fichiers de géométries GeoJSON et les rasters climatologiques WorldClim GeoTIFF ont été inspectés, validés, et sont pleinement fonctionnels pour alimenter le moteur physique global.